# Casino Player Analytics Pipeline

This notebook walks through all four sections of the pipeline using the casino_analytics package.
All data is synthetic (generated by synth/generate.py). Run the setup cell first.

Sections:
1. Feature Engineering
2. Trend Analytics and Segmentation
3. Demographic Distributions
4. Propensity-Score Matching + Difference-in-Differences

In [ ]:
# Setup — add src/ to path so the package is importable without installing
import sys, pathlib
repo = pathlib.Path().resolve().parent
sys.path.insert(0, str(repo / 'src'))
sys.path.insert(0, str(repo))
print('Repo root:', repo)

---
## Section 1 | Feature Engineering
### 1.1 Load and clean raw tables

In [ ]:
from casino_analytics.loaders import load_all
from casino_analytics.cleaning import clean_players, parse_play_dates

tables = load_all()
play = parse_play_dates(tables['play'])
players = clean_players(tables['players'])
app_events = tables['app_events']

print(f'play:       {len(play):>8,} rows')
print(f'players:    {len(players):>8,} rows (deduplicated)')
print(f'jackpots:   {len(tables["jackpots"]):>8,} rows')
print(f'app_events: {len(app_events):>8,} rows')

### 1.2 Build daily table with all feature columns

In [ ]:
from casino_analytics.features.daily import merge_demographics, aggregate_daily
from casino_analytics.features.demographics import add_demographics
from casino_analytics.features.financials import add_financials
from casino_analytics.features.geo import add_geo

merged = merge_demographics(play, players)
daily = aggregate_daily(merged)
daily = add_demographics(daily)
daily = add_financials(daily)
daily = add_geo(daily)

print(f'daily: {daily.shape[0]:,} rows x {daily.shape[1]} columns')
daily[['player_id','accounting_date','coin_in','theo_hold_pct','act_hold_pct',
       'age_at_visit','age_bucket','distance_miles','mileage_band']].head()

### 1.3 Player rollup and player-month panel

In [ ]:
from casino_analytics.features.player import build_player_rollup, build_player_month_panel
from casino_analytics.segmentation import assign_segments

player = build_player_rollup(daily)
player = assign_segments(player)
panel = build_player_month_panel(daily)

print(f'player: {len(player):,} rows')
print('L1 segments:')
print(player['l1_segment'].value_counts().to_string())
print()
print(f'panel:  {len(panel):,} rows')

---
## Section 2 | Trend Analytics
### 2.1 Monthly Active Users by segment

In [ ]:
from casino_analytics.analytics.trends import monthly_active_users, yoy_comparison
from casino_analytics.plots import plot_mau_trend, plot_mau_by_segment
from IPython.display import Image

mau_seg = monthly_active_users(panel, player)
mau_total = (
    mau_seg.groupby(['year','month','period'])
    .agg(mau=('mau','sum'), total_coin_in=('total_coin_in','sum'))
    .reset_index()
)
yoy = yoy_comparison(mau_total)

p1 = plot_mau_trend(mau_total)
p2 = plot_mau_by_segment(mau_seg)
print(f'Peak MAU: {mau_total["mau"].max():,} ({mau_total.loc[mau_total["mau"].idxmax(), "period"].strftime("%b %Y")})')
Image(p1)

### 2.2 Year-over-Year comparison

In [ ]:
yoy[['year','month','mau','mau_yoy_pct','coin_yoy_pct']].dropna().tail(12)

---
## Section 3 | Demographic Distributions
### 3.1 Age distribution

In [ ]:
from casino_analytics.plots import plot_age_distribution

p3 = plot_age_distribution(player)
Image(p3)

### 3.2 Enrollment over time

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

enroll = player.copy()
enroll['enroll_year'] = enroll['enrollment_date'].dt.year
counts = enroll.groupby('enroll_year').size().reset_index(name='n_players')

fig, ax = plt.subplots(figsize=(7,3))
ax.bar(counts['enroll_year'], counts['n_players'], color='#2563EB')
ax.set_title('Player Enrollment by Year (synthetic data)')
ax.set_xlabel('Year'); ax.set_ylabel('New Players')
plt.tight_layout(); plt.show()
print('Note: all values are synthetic')

---
## Section 4 | Causal Inference
### 4.0 Section 4 Parameters

In [ ]:
# Causal section parameters
FEATURE_COLS = ['amv', 'adt', 'n_visit_days', 'distance_miles']
TREATMENT_YEAR  = 2024
TREATMENT_MONTH = 4    # April 2024 = start of post period
MAX_PRE  = 6
MAX_POST = 6
PSM_CALIPER = 0.05
N_CONTROLS  = 5

### 4.1 Define treatment and build analytic sample

In [ ]:
treated_ids = set(app_events['player_id'])
player['treated'] = player['player_id'].isin(treated_ids).astype(int)

print(f'Treated:  {player["treated"].sum():,}')
print(f'Control:  {(player["treated"]==0).sum():,}')
print(f'Share:    {player["treated"].mean():.1%}')

### 4.2 Pre-treatment summary statistics

In [ ]:
player[FEATURE_COLS + ['treated']].groupby('treated').mean().round(2)

### 4.3 Estimate propensity scores

In [ ]:
from casino_analytics.causal.psm import estimate_propensity, trim_common_support, match_nearest_neighbour, balance_table

player_clean = player[FEATURE_COLS + ['player_id','treated']].dropna()
player_ps = estimate_propensity(player_clean, FEATURE_COLS)
player_trim = trim_common_support(player_ps)

print(f'Before trim: {len(player_ps):,}  After trim: {len(player_trim):,}')
player_ps[['treated','propensity']].groupby('treated').describe().round(3)

### 4.4 Visualise propensity score overlap

In [ ]:
from casino_analytics.plots import plot_propensity_overlap
matched = match_nearest_neighbour(player_trim, n_controls=N_CONTROLS, caliper=PSM_CALIPER)
print(f'Matched: {matched["treated"].sum():,} treated, {(matched["treated"]==0).sum():,} controls')
p4 = plot_propensity_overlap(matched)
Image(p4)

### 4.5 Covariate balance before and after matching

In [ ]:
from casino_analytics.plots import plot_balance
btable = balance_table(player_trim, matched, FEATURE_COLS)
p5 = plot_balance(btable)
Image(p5)

### 4.6 DiD regressions on matched sample

In [ ]:
from casino_analytics.causal.did import add_post_indicator, did_basic, did_with_month_fe

panel_m = panel.merge(player_clean[['player_id','treated']], on='player_id', how='inner')
panel_m = add_post_indicator(panel_m, TREATMENT_YEAR, TREATMENT_MONTH)

basic  = did_basic(panel_m)
fe_res = did_with_month_fe(panel_m)

att_basic = basic.params.get('treat_post', float('nan'))
att_fe    = fe_res.params.get('treat_post', float('nan'))

print(f'ATT (basic DiD)            : {att_basic:+.2f}  (p={basic.pvalues.get("treat_post",float("nan")):.3f})')
print(f'ATT (DiD + month FE)       : {att_fe:+.2f}  (p={fe_res.pvalues.get("treat_post",float("nan")):.3f})')
print('(All on synthetic data only)')

### 4.7 Event study and parallel trends test

In [ ]:
from casino_analytics.causal.did import build_event_study_panel, event_study, parallel_trends_f_test
from casino_analytics.plots import plot_event_study

es_panel = build_event_study_panel(panel_m, app_events, max_pre=MAX_PRE, max_post=MAX_POST)
es_coefs = event_study(es_panel)
pt = parallel_trends_f_test(es_panel)

print(f'Parallel trends F-test: F={pt["f_stat"]}, p={pt["p_value"]} '
      f'({"PASS" if pt.get("passes_at_05") else "FAIL"} at alpha=0.05)')

p6 = plot_event_study(es_coefs)
Image(p6)